# Games to Delta table

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, IntegerType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import sys
sys.path.insert(0, "/Workspace/Repos")
from transforms import conform_games

ACCOUNT    = "footballmanagerli"
SRC        = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/games/"
DEST       = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/games/games"
CHECKPOINT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/games"
TABLE      = "nfl.games"

In [ ]:
game_schema = StructType([
    StructField("game_id",      StringType()),
    StructField("date",         StringType()),
    StructField("status",       StringType()),
    StructField("completed",    BooleanType()),
    StructField("venue",        StringType()),
    StructField("city",         StringType()),
])

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nfl")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    game_id        STRING    COMMENT 'Game ID (primary key)',
    date           STRING    COMMENT 'Game date (ISO 8601)',
    status         STRING    COMMENT 'Game status (e.g., Final, In Progress)',
    completed      BOOLEAN   COMMENT 'Whether game has completed',
    venue          STRING    COMMENT 'Game venue',
    city           STRING    COMMENT 'Venue city',
    _source_file   STRING    COMMENT 'Blob file path',
    _ingested_at   TIMESTAMP COMMENT 'When ingested into Delta'
)
USING DELTA
LOCATION '{DEST}'
""")

In [ ]:
raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .schema(game_schema)
    .load(SRC)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time").alias("_snapshot_at"),
    ))

# Apply conform function - outputs exactly target_schemas['games'] columns
games = conform_games(raw).withColumn(
    "_ingested_at", F.current_timestamp()
)

In [ ]:
def upsert_batch(batch_df, batch_id):
    """Upsert with automatic schema evolution: add missing columns if source has new fields."""
    
    # Step 1: Dedupe within batch (keep newest snapshot per game)
    w = Window.partitionBy("game_id").orderBy(F.col("_snapshot_at").desc())
    latest = (batch_df
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn", "_snapshot_at"))
    
    # Step 2: Auto-add missing columns (schema evolution)
    target_table = DeltaTable.forName(spark, TABLE)
    target_schema = {field.name: field.dataType for field in target_table.toDF().schema}
    batch_columns = {field.name for field in latest.schema}
    
    for col_name, col_type in target_schema.items():
        if col_name not in batch_columns:
            latest = latest.withColumn(col_name, F.lit(None).cast(col_type))
    
    # Step 3: Merge into table
    (target_table.alias("t")
        .merge(latest.alias("s"), "t.game_id = s.game_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

(games.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

In [ ]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT game_id) AS unique_games FROM nfl.games;

SELECT date, venue, city, completed FROM nfl.games ORDER BY date DESC LIMIT 20;